In [ ]:
#!pip uninstall numpy -y

In [ ]:
#!pip install numpy==1.26.4

In [ ]:

!pip install torchmetrics
!pip install moviepy pydub
import torch as pt
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
import torchmetrics
import librosa
import numpy as np
import matplotlib.pyplot as plt
import sys
import random
import os
import tqdm as tqdm
import time
from google.colab import drive
drive.mount('/content/drive')

# Используем кэширование
from joblib import Memory
cachedir = '/content/cache/'
memory = Memory(cachedir, verbose=0, compress=True)
memory.clear(warn=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 14.3 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
# @title
# Для воспроизводимости результатов
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
pt.manual_seed(random_seed)
pt.cuda.manual_seed(random_seed)
pt.cuda.manual_seed_all(random_seed)
pt.backends.cudnn.deterministic = True

device = 'cuda' if pt.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

!unzip -u "drive/MyDrive/AudioDataset_Detection_Balanced.zip" -d "/content/dataset"
dataset_directory = '/content/dataset/AudioDataset_Detection_Balanced'
duration = 0.5
BATCH_SIZE = 64
NUM_EPOCHS = 18

def preprocess_file(filename, duration):
    waveform, sr = librosa.load(filename, sr=44100, duration=duration)
    waveform = normalize(waveform)
    spec = melspectrogram(waveform, sr)
    return spec.astype(np.float32)  # гарантирует float32

def normalize(waveform):
    s = waveform.astype(np.float32) - np.mean(waveform)
    std = np.std(s)
    return s / (std + 1e-8)

def melspectrogram(waveform, sr):
    return librosa.feature.melspectrogram(
        y=waveform, sr=sr, n_fft=1024, hop_length=512, n_mels=64, pad_mode='constant'
    )

class DroneDataset(Dataset):
    def __init__(self, dataset_dir, duration, device):
        files_list = []
        for label_name, label in [("Background", 0), ("Drone", 1)]:
            folder = os.path.join(dataset_dir, label_name)
            for fname in os.listdir(folder):
                if fname.endswith(".wav"):
                    files_list.append((os.path.join(folder, fname), label))
        self.datalist = files_list
        self.duration = duration
        self.device = device

    def __len__(self):
        return len(self.datalist)

    def __getitem__(self, idx):
        spec = preprocess_file(self.datalist[idx][0], self.duration)
        spec = pt.from_numpy(spec[None, ...]).to(self.device)
        # Исправление: метка как float32 тензор
        label = pt.tensor(float(self.datalist[idx][1]), dtype=pt.float32)
        return spec, label

train_dataset = DroneDataset(os.path.join(dataset_directory, "train"), duration, device)
test_dataset = DroneDataset(os.path.join(dataset_directory, "test"), duration, device)
val_dataset = DroneDataset(os.path.join(dataset_directory, "val"), duration, device)

train_data_size = len(train_dataset)
val_data_size = len(val_dataset)
test_data_size = len(test_dataset)
print(train_data_size, val_data_size, test_data_size)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

class CNNNetwork(nn.Module):
    def __init__(self, dropout_rate=0.2):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.Dropout2d(dropout_rate)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.Dropout2d(dropout_rate)
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.conv5 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.Dropout2d(dropout_rate)
        )
        self.conv6 = nn.Sequential(
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.conv7 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.Dropout2d(dropout_rate)
        )
        self.conv8 = nn.Sequential(
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d((4, 4))
        )

        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        x = self.conv6(x)
        x = self.conv7(x)
        x = self.conv8(x)
        x = self.flatten(x)
        return self.classifier(x)

model = CNNNetwork().to(device)
summary(model, (1, 64, 44))

loss_func = nn.BCELoss()
optimizer = pt.optim.Adam(model.parameters(), lr=0.0001)

# Создаем метрики
accuracy_metric = torchmetrics.classification.Accuracy(task="binary", threshold=0.5).to(device)
precision_metric = torchmetrics.classification.Precision(task="binary", threshold=0.5).to(device)
recall_metric = torchmetrics.classification.Recall(task="binary", threshold=0.5).to(device)
f1_metric = torchmetrics.classification.F1Score(task="binary", threshold=0.5).to(device)

def compute_metrics(y_pred, y):
    """Вычисление всех метрик"""
    accuracy = accuracy_metric(y_pred, y)
    precision = precision_metric(y_pred, y)
    recall = recall_metric(y_pred, y)
    f1 = f1_metric(y_pred, y)
    return accuracy, precision, recall, f1

def train_and_validate(model, loss_criterion, optimizer, epochs=5):
    history = []
    best_loss = float('inf')
    best_epoch = -1

    for epoch in range(epochs):
        # Тренировочная эпоха
        model.train()
        train_loss = 0.0
        train_accuracy = 0.0
        train_precision = 0.0
        train_recall = 0.0
        train_f1 = 0.0

        # Сбрасываем метрики перед началом эпохи
        accuracy_metric.reset()
        precision_metric.reset()
        recall_metric.reset()
        f1_metric.reset()

        for inputs, labels in train_dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs).squeeze(-1)
            loss = loss_criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)

            # Вычисляем метрики
            acc, prec, rec, f1 = compute_metrics(outputs, labels)
            train_accuracy += acc.item() * inputs.size(0)
            train_precision += prec.item() * inputs.size(0)
            train_recall += rec.item() * inputs.size(0)
            train_f1 += f1.item() * inputs.size(0)

        # Валидационная эпоха
        model.eval()
        val_loss = 0.0
        val_accuracy = 0.0
        val_precision = 0.0
        val_recall = 0.0
        val_f1 = 0.0

        # Сбрасываем метрики перед валидацией
        accuracy_metric.reset()
        precision_metric.reset()
        recall_metric.reset()
        f1_metric.reset()

        with pt.no_grad():
            for inputs, labels in val_dataloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs).squeeze(-1)
                loss = loss_criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)

                # Вычисляем метрики
                acc, prec, rec, f1 = compute_metrics(outputs, labels)
                val_accuracy += acc.item() * inputs.size(0)
                val_precision += prec.item() * inputs.size(0)
                val_recall += rec.item() * inputs.size(0)
                val_f1 += f1.item() * inputs.size(0)

        # Вычисляем средние значения
        avg_train_loss = train_loss / train_data_size
        avg_val_loss = val_loss / val_data_size

        avg_train_acc = train_accuracy / train_data_size
        avg_val_acc = val_accuracy / val_data_size

        avg_train_prec = train_precision / train_data_size
        avg_val_prec = val_precision / val_data_size

        avg_train_rec = train_recall / train_data_size
        avg_val_rec = val_recall / val_data_size

        avg_train_f1 = train_f1 / train_data_size
        avg_val_f1 = val_f1 / val_data_size

        # Сохраняем лучшую модель
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            best_epoch = epoch
            pt.save(model.state_dict(), f'best_model_epoch_{epoch}.pt')

        # Сохраняем историю
        history.append([
            avg_train_loss, avg_val_loss,
            avg_train_acc, avg_val_acc,
            avg_train_prec, avg_val_prec,
            avg_train_rec, avg_val_rec,
            avg_train_f1, avg_val_f1
        ])

        # Выводим метрики
        print(f"Epoch {epoch+1}/{epochs}:")
        print(f"  Train - Loss: {avg_train_loss:.4f}, Acc: {avg_train_acc:.4f}, "
              f"Precision: {avg_train_prec:.4f}, Recall: {avg_train_rec:.4f}, F1: {avg_train_f1:.4f}")
        print(f"  Val   - Loss: {avg_val_loss:.4f}, Acc: {avg_val_acc:.4f}, "
              f"Precision: {avg_val_prec:.4f}, Recall: {avg_val_rec:.4f}, F1: {avg_val_f1:.4f}")
        print("-" * 80)

    return model, history, best_epoch

trained_model, history, best_epoch = train_and_validate(model, loss_func, optimizer, NUM_EPOCHS)

# Тестирование модели
def test_model(model, test_loader):
    model.eval()
    test_accuracy = 0.0
    test_precision = 0.0
    test_recall = 0.0
    test_f1 = 0.0

    # Сбрасываем метрики перед тестированием
    accuracy_metric.reset()
    precision_metric.reset()
    recall_metric.reset()
    f1_metric.reset()

    all_preds = []
    all_labels = []

    with pt.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs).squeeze(-1)

            # Собираем предсказания и метки
            preds = (outputs > 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            # Вычисляем метрики
            acc, prec, rec, f1 = compute_metrics(outputs, labels)
            test_accuracy += acc.item() * inputs.size(0)
            test_precision += prec.item() * inputs.size(0)
            test_recall += rec.item() * inputs.size(0)
            test_f1 += f1.item() * inputs.size(0)

    avg_test_acc = test_accuracy / test_data_size
    avg_test_prec = test_precision / test_data_size
    avg_test_rec = test_recall / test_data_size
    avg_test_f1 = test_f1 / test_data_size

    print("\n" + "="*80)
    print("ТЕСТИРОВАНИЕ МОДЕЛИ:")
    print(f"Accuracy:  {avg_test_acc:.4f}")
    print(f"Precision: {avg_test_prec:.4f}")
    print(f"Recall:    {avg_test_rec:.4f}")
    print(f"F1-Score:  {avg_test_f1:.4f}")
    print("="*80)

    return avg_test_acc, avg_test_prec, avg_test_rec, avg_test_f1, all_preds, all_labels

# Запускаем тестирование
test_results = test_model(trained_model, test_dataloader)

Using device: cpu
Archive:  drive/MyDrive/AudioDataset_Detection_Balanced.zip
6590 942 1884
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 64, 44]             320
       BatchNorm2d-2           [-1, 32, 64, 44]              64
              ReLU-3           [-1, 32, 64, 44]               0
         Dropout2d-4           [-1, 32, 64, 44]               0
            Conv2d-5           [-1, 32, 64, 44]           9,248
       BatchNorm2d-6           [-1, 32, 64, 44]              64
              ReLU-7           [-1, 32, 64, 44]               0
         MaxPool2d-8           [-1, 32, 32, 22]               0
            Conv2d-9           [-1, 64, 32, 22]          18,496
      BatchNorm2d-10           [-1, 64, 32, 22]             128
             ReLU-11           [-1, 64, 32, 22]               0
        Dropout2d-12           [-1, 64, 32, 22]               0
           

In [ ]:
best_model = pt.load(f"best_model_epoch_{best_epoch}.pt", weights_only=False)
best_model = trained_model

# === ОЦЕНКА НА ТЕСТОВОМ НАБОРЕ ===
true_labels = []
pred_labels = []

def computeTestSetAccuracy(model, loss_criterion):
    model.eval()
    test_acc = test_loss = 0.0
    with pt.no_grad():
        for inputs, labels in test_dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs).squeeze(-1)
            true_labels.extend(labels.cpu().numpy().astype(int))
            pred_labels.extend((outputs > 0.5).cpu().numpy().astype(int))
            loss = loss_criterion(outputs, labels)
            test_loss += loss.item() * inputs.size(0)
            test_acc += compute_accuracy(outputs, labels).item() * inputs.size(0)
    avg_test_acc = test_acc / test_data_size
    avg_test_loss = test_loss / test_data_size
    print(f"Test Accuracy: {avg_test_acc:.5f}, Test Loss: {avg_test_loss:.5f}")
    return avg_test_acc

# Выполняем оценку один раз
original_acc = computeTestSetAccuracy(best_model, loss_func)

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
print('F1: {:.5f}'.format(f1_score(true_labels, pred_labels, average="binary")))
print('Accuracy: {:.5f}'.format(accuracy_score(true_labels, pred_labels)))
print('Precision: {:.5f}'.format(precision_score(true_labels, pred_labels, average="binary")))
print('Recall: {:.5f}'.format(recall_score(true_labels, pred_labels, average="binary")))

def add_white_noise_proper(signal, snr_db):
    """Добавление белого шума с заданным SNR в dB"""
    signal_power = np.mean(signal**2)
    snr_linear = 10**(snr_db / 10)
    noise_power = signal_power / snr_linear
    noise = np.random.normal(0, np.sqrt(noise_power), len(signal))
    return signal + noise
def add_background_noise_proper(signal, sr, snr_db, background_files, duration):
    """Добавление фонового шума с заданным SNR в dB"""
    if not background_files:
        return signal

    bg_file = np.random.choice(background_files)
    bg_signal, _ = librosa.load(bg_file, sr=sr, duration=duration)

    # Выравнивание длин
    if len(bg_signal) < len(signal):
        bg_signal = np.pad(bg_signal, (0, len(signal) - len(bg_signal)), mode='constant')
    else:
        bg_signal = bg_signal[:len(signal)]

    # Расчет мощностей и масштабирование
    signal_power = np.mean(signal**2)
    bg_power = np.mean(bg_signal**2)

    if bg_power == 0:  # защита от деления на ноль
        return signal

    snr_linear = 10**(snr_db / 10)
    desired_bg_power = signal_power / snr_linear
    scale_factor = np.sqrt(desired_bg_power / bg_power)

    return signal + bg_signal * scale_factor

def calculate_real_snr(clean_signal, noisy_signal):
    """Расчет реального SNR между чистым и зашумленным сигналом"""
    signal_power = np.mean(clean_signal**2)
    noise_power = np.mean((clean_signal - noisy_signal)**2)
    if noise_power == 0:
        return float('inf')
    return 10 * np.log10(signal_power / noise_power)

def combined_noise(signal, sr, snr_db, background_files, pitch_steps=2.0, time_shift_sec=0.1):
    """
    Комбинированный шум с правильным расчетом SNR

        signal: исходный сигнал
        sr: частота дискретизации
        snr_db: SNR в dB для аддитивных шумов
        background_files: список фоновых файлов
        pitch_steps: сдвиг высоты тона в полутонах
        time_shift_sec: временной сдвиг в секундах
    """
    # Добавляем белый шум с правильным SNR
    signal = add_white_noise_proper(signal, snr_db)

    # Добавляем фоновый шум с тем же SNR
    if background_files and len(background_files) > 0:
        signal = add_background_noise_proper(signal, sr, snr_db, background_files, len(signal)/sr)

    return signal

# Загрузка фоновых файлов
background_files = []
bg_dir = os.path.join(dataset_directory, "train", "Background")
for fname in os.listdir(bg_dir):
    if fname.endswith(".wav"):
        background_files.append(os.path.join(bg_dir, fname))

class NoisyDroneDataset(Dataset):
    def __init__(self, original_dataset, noise_type, noise_param, sr=44100):
        self.orig = original_dataset
        self.noise_type = noise_type
        self.noise_param = noise_param
        self.sr = sr
        self.bg_files = background_files

    def __len__(self):
        return len(self.orig)

    def __getitem__(self, idx):
        path, label = self.orig.datalist[idx]
        clean_wave, _ = librosa.load(path, sr=self.sr, duration=duration)
        clean_wave = normalize(clean_wave)

        if self.noise_type == 'white':
            noisy_wave = add_white_noise_proper(clean_wave, self.noise_param)
        elif self.noise_type == 'background':
            noisy_wave = add_background_noise_proper(clean_wave, self.sr, self.noise_param,
                                                   self.bg_files, duration)
        elif self.noise_type == 'combined':
            noisy_wave = combined_noise(
                clean_wave,
                sr=self.sr,
                snr_db=self.noise_param,
                background_files=self.bg_files,
            )

        else:
            raise ValueError(f"Unknown noise type: {self.noise_type}")


        spec = melspectrogram(noisy_wave, self.sr).astype(np.float32)
        spec = pt.from_numpy(spec[None, ...]).to(device)
        label = pt.tensor(float(label), dtype=pt.float32)
        return spec, label

def calculate_robustness_metrics(original_acc, noisy_accuracies, noise_params):
    """Расчет метрик устойчивости"""
    accuracy_drop = [original_acc - acc for acc in noisy_accuracies]
    robustness_threshold = None

    for i, acc in enumerate(noisy_accuracies):
        if acc < 0.7:
            robustness_threshold = noise_params[i]
            break

    return {
        'accuracy_drop': accuracy_drop,
        'robustness_threshold': robustness_threshold,
        'mean_accuracy_drop': np.mean(accuracy_drop)
    }

def test_model_on_noisy_data(model, test_dataset, noise_type, noise_level):
    noisy_ds = NoisyDroneDataset(test_dataset, noise_type, noise_level)
    loader = DataLoader(noisy_ds, batch_size=BATCH_SIZE, shuffle=False)
    model.eval()
    true, pred = [], []
    with pt.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            out = model(inputs).squeeze(-1)
            preds = (out > 0.5).float().cpu().numpy()
            true.extend(labels.numpy().astype(int))
            pred.extend(preds.astype(int))
    return accuracy_score(true, pred)

WHITE_NOISE_LEVELS = [3,2,1,0,-1,-2,-3,-4,-5]
BACKGROUND_NOISE_LEVELS = [5,4,3,2,1,0,-1,-2,-3,-4,-5]
COMBINED_LEVELS = [5,4,3,2,1,0,-1,-2,-3,-4,-5]

print(f"\n=== Оригинал: Accuracy = {original_acc:.5f} ===")

print("\n" + "="*80)
print(f"{'Тип шума':<15} {'Параметр':<10} {'Accuracy':<12} {'Потеря Accuracy':<15}")
print("-"*80)
print(f"{'ОРИГИНАЛ':<15} {'-':<10} {original_acc:<12.5f} {'-':<15}")

white_accuracies = []
for snr in WHITE_NOISE_LEVELS:
    acc = test_model_on_noisy_data(best_model, test_dataset, 'white', snr)
    white_accuracies.append(acc)
    loss = original_acc - acc
    print(f"{'Белый шум':<15} {f'{snr} dB':<10} {acc:<12.5f} {loss:<15.5f}")

white_metrics = calculate_robustness_metrics(original_acc, white_accuracies, WHITE_NOISE_LEVELS)
print(f"Порог устойчивости к белому шуму: {white_metrics['robustness_threshold']} dB")

background_accuracies = []
for snr in BACKGROUND_NOISE_LEVELS:
    acc = test_model_on_noisy_data(best_model, test_dataset, 'background', snr)
    background_accuracies.append(acc)
    loss = original_acc - acc
    print(f"{'Фоновый шум':<15} {f'{snr} dB':<10} {acc:<12.5f} {loss:<15.5f}")

background_metrics = calculate_robustness_metrics(original_acc, background_accuracies, BACKGROUND_NOISE_LEVELS)
print(f"Порог устойчивости к фоновому шуму: {background_metrics['robustness_threshold']} dB")

combined_accuracies = []
for snr in COMBINED_LEVELS:
    acc = test_model_on_noisy_data(best_model, test_dataset, 'combined', snr)
    combined_accuracies.append(acc)
    loss = original_acc - acc
    print(f"{'Combined':<15} {f'{snr} dB':<10} {acc:<12.5f} {loss:<15.5f}")

combined_metrics = calculate_robustness_metrics(original_acc, combined_accuracies, COMBINED_LEVELS)
print(f"Порог устойчивости к комбинированному шуму: {combined_metrics['robustness_threshold']} dB")

plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
plt.plot(WHITE_NOISE_LEVELS, white_accuracies, 'bo-', label='Белый шум', linewidth=2)
plt.plot(BACKGROUND_NOISE_LEVELS, background_accuracies, 'ro-', label='Фоновый шум', linewidth=2)
plt.plot(COMBINED_LEVELS, combined_accuracies, 'co-', label='Комбинированный', linewidth=2)
plt.axhline(y=original_acc, color='g', linestyle='--', label='Исходная точность')
plt.axhline(y=0.7, color='orange', linestyle=':', label='Порог полезности (0.7)')
plt.xlabel('SNR (dB)')
plt.ylabel('Accuracy')
plt.title('Устойчивость к аддитивным шумам')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 4)
losses = [
    np.mean([original_acc - acc for acc in white_accuracies]),
    np.mean([original_acc - acc for acc in background_accuracies]),
    np.mean([original_acc - acc for acc in combined_accuracies])
]
types = ['Белый\nшум', 'Фоновый\nшум', 'Комбини-\nрованный']
plt.bar(types, losses, color=['blue', 'red', 'cyan'], alpha=0.7)
plt.ylabel('Средняя потеря точности')
plt.title('Сравнение устойчивости к разным типам шумов')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('robustness_analysis_detection.png', dpi=200, bbox_inches='tight')
plt.show()

print("\n=== Пример предсказания на одном сэмпле ===")
waveform, true_label = test_dataset[0]
best_model.eval()
with pt.no_grad():
    inputs = waveform.unsqueeze(0)
    pred = best_model(inputs)
    prob = pred.item()
    predicted_class = 1 if prob > 0.5 else 0
print(f"Истинная метка: {int(true_label)}")
print(f"Предсказанная вероятность: {prob:.5f}")
print(f"Предсказанный класс: {predicted_class}")

print("\nТестирование устойчивости к шуму завершено.")

In [ ]:
# =============================================
# БЛОК АНАЛИЗА ВИДЕО ЧЕРЕЗ АУДИО
# =============================================

def extract_audio_from_video(video_path, output_audio_path):
    """Извлечение аудиодорожки из видео"""
    try:
        from moviepy.editor import VideoFileClip
        video = VideoFileClip(video_path)
        audio = video.audio
        audio.write_audiofile(output_audio_path, verbose=False, logger=None)
        video.close()
        return True
    except Exception as e:
        print(f"Ошибка извлечения аудио: {e}")
        return False

def analyze_audio_for_drones(audio_path, model, duration=0.5, overlap=0.25):
    """
    Анализ аудиофайла на наличие дронов с скользящим окном
    """
    # Загружаем аудио
    audio, sr = librosa.load(audio_path, sr=44100)

    # Параметры окон
    window_size = int(duration * sr)
    hop_size = int((duration - overlap) * sr)

    results = []
    timestamps = []

    # Обрабатываем аудио окнами
    for start in range(0, len(audio) - window_size, hop_size):
        end = start + window_size
        audio_chunk = audio[start:end]

        # Препроцессинг как в обучении
        audio_chunk = normalize(audio_chunk)
        spec = melspectrogram(audio_chunk, sr).astype(np.float32)
        spec = pt.from_numpy(spec[None, None, ...]).to(device)  # [1, 1, 64, 44]

        # Предсказание
        with pt.no_grad():
            prediction = model(spec).item()

        timestamp = start / sr
        results.append(prediction)
        timestamps.append(timestamp)

    return {
        'timestamps': timestamps,
        'predictions': results,
        'audio_length': len(audio) / sr
    }

def process_video_for_drone_detection(video_path, model, duration=0.5, overlap=0.25):
    """
    Полный пайплайн: видео -> аудио -> анализ нейросетью
    """
    # Временный файл для аудио
    temp_audio = "temp_audio.wav"

    # 1. Извлекаем аудио
    print("Извлекаем аудио из видео...")
    if not extract_audio_from_video(video_path, temp_audio):
        return None

    # 2. Анализируем аудио
    print("Анализируем аудио на наличие дронов...")
    results = analyze_audio_for_drones(temp_audio, model, duration, overlap)

    # 3. Удаляем временный файл
    import os
    if os.path.exists(temp_audio):
        os.remove(temp_audio)

    return results

def plot_drone_detection_results(results, threshold=0.5):
    """
    Визуализация результатов детекции дронов во времени
    """
    timestamps = results['timestamps']
    predictions = results['predictions']

    plt.figure(figsize=(15, 8))

    # График вероятностей
    plt.subplot(2, 1, 1)
    plt.plot(timestamps, predictions, 'b-', alpha=0.7, label='Вероятность дрона')
    plt.axhline(y=threshold, color='r', linestyle='--', label=f'Порог ({threshold})')
    plt.fill_between(timestamps, 0, predictions,
                    where=[p > threshold for p in predictions],
                    color='red', alpha=0.3, label='Дрон обнаружен')
    plt.ylabel('Вероятность')
    plt.title('Детекция дронов по аудио из видео')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Бинарная детекция
    plt.subplot(2, 1, 2)
    binary_detection = [1 if p > threshold else 0 for p in predictions]
    plt.step(timestamps, binary_detection, 'r-', where='post', linewidth=2)
    plt.ylabel('Дрон (1 - да, 0 - нет)')
    plt.xlabel('Время (секунды)')
    plt.yticks([0, 1])
    plt.grid(True, alpha=0.3)

    # Статистика
    total_time = results['audio_length']
    drone_time = sum(binary_detection) * (timestamps[1] - timestamps[0])
    drone_percentage = (drone_time / total_time) * 100

    plt.figtext(0.02, 0.02,
                f"Общее время: {total_time:.1f}с | "
                f"Время с дроном: {drone_time:.1f}с ({drone_percentage:.1f}%)",
                fontsize=12, bbox=dict(boxstyle="round", facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.savefig('video_audio_detection.png', dpi=200, bbox_inches='tight')
    plt.show()

    return drone_percentage

def smooth_predictions(predictions, window_size=5):
    """Сглаживание предсказаний скользящим средним"""
    return np.convolve(predictions, np.ones(window_size)/window_size, mode='same')

def advanced_detection_logic(predictions, timestamps, threshold=0.5, min_duration=1.0):
    """
    Продвинутая логика детекции:
    - Игнорировать короткие срабатывания
    - Объединять близкие детекции
    """
    # Сглаживание
    smoothed = smooth_predictions(predictions)

    # Бинаризация
    binary = [1 if p > threshold else 0 for p in smoothed]

    # Поиск сегментов
    segments = []
    in_segment = False
    segment_start = 0

    for i, (ts, det) in enumerate(zip(timestamps, binary)):
        if det and not in_segment:
            in_segment = True
            segment_start = ts
        elif not det and in_segment:
            in_segment = False
            segment_end = ts
            duration = segment_end - segment_start
            if duration >= min_duration:
                segments.append((segment_start, segment_end, duration))

    return segments

In [ ]:
print("\n" + "="*60)
print("БЛОК АНАЛИЗА ВИДЕО ЧЕРЕЗ АУДИО")
print("="*60)

best_model.eval()

video_path = "/content/drive/MyDrive/video/drone_video_mav1.mp4"

try:
    print(f"Пытаемся проанализировать видео: {video_path}")
    results = process_video_for_drone_detection(video_path, best_model)

    if results:
        drone_percentage = plot_drone_detection_results(results)
        print(f"РЕЗУЛЬТАТ: Дрон обнаружен в {drone_percentage:.1f}% времени видео")

        segments = advanced_detection_logic(results['predictions'], results['timestamps'])
        print(f"Найдено сегментов с дроном: {len(segments)}")
        for i, (start, end, duration) in enumerate(segments):
            print(f"  Сегмент {i+1}: {start:.1f}с - {end:.1f}с (длительность: {duration:.1f}с)")
    else:
        print("Не удалось обработать видео")

except Exception as e:
    print(f"Ошибка при обработке видео: {e}")

print("\nАнализ видео завершен!")